# Task 1: Data Cleaning & Feature Engineering

This notebook merges historical match data with supplementary team statistics, handles missing values, and engineers features necessary for modeling.

In [1]:
import pandas as pd
import numpy as np
import os

# Load data
matches_df = pd.read_csv('../data/raw/historical_matches.csv')
stats_df = pd.read_csv('../data/raw/supplementary_stats.csv')

print("Matches shape:", matches_df.shape)
print("Stats shape:", stats_df.shape)

Matches shape: (53394, 9)
Stats shape: (49, 5)


## Data Cleaning
Remove duplicates and handle missing values.

In [2]:
# Drop duplicates
matches_df = matches_df.drop_duplicates()

# Focus on post-1990 data for relevance
matches_df['date'] = pd.to_datetime(matches_df['date'])
matches_df = matches_df[matches_df['date'].dt.year >= 1990]

# Fill missing scores if any
matches_df['home_score'] = matches_df['home_score'].fillna(0)
matches_df['away_score'] = matches_df['away_score'].fillna(0)


## Feature Engineering
Engineer goal difference, team win rate, and merge with average age/caps.

In [3]:
# Goal difference
matches_df['goal_diff'] = matches_df['home_score'] - matches_df['away_score']

# Determine winner (1 for Home, 0 for Draw, -1 for Away)
matches_df['result'] = np.where(matches_df['goal_diff'] > 0, 1, 
                                np.where(matches_df['goal_diff'] < 0, -1, 0))

# Target variable for classification: Did Home Team Win? (1/0)
matches_df['home_win'] = (matches_df['result'] == 1).astype(int)

# Merge with stats (Simulated: assuming home team stats for simplicity of this baseline)
# In a robust model, we would merge both home and away stats and compute the difference.
final_df = pd.merge(matches_df, stats_df, left_on='home_team', right_on='team', how='inner')
final_df = final_df.drop(columns=['team'])

os.makedirs('../data/processed', exist_ok=True)
final_df.to_csv('../data/processed/final_features.csv', index=False)
print("Saved final_features.csv with shape:", final_df.shape)


Saved final_features.csv with shape: (11259, 16)
